# M3-B1 — Mission étoile : préparer la 3ᵉ source pour du RAG — SUJET

> **Bonus optionnel, non noté.** Acerox t'a transmis un corpus de **rapports
> techniques** (`rapports_techniques_2024.zip`, 5 fichiers `.md`). C'est du
> **texte non structuré** : impossible à ranger dans une table SQL. Tu vas le
> **préparer** pour une exploitation par similarité sémantique.

## 🚫 Garde-fou
> **Préparation seulement** : tu t'arrêtes à la **récupération** (retrieval).
> La génération augmentée par un LLM, c'est **M7-B2**. Pour la prédiction de
> défauts (tabulaire), un RAG **n'aide pas** : ce corpus sert l'**aide au
> diagnostic humain**. Embeddings **locaux** (aucune clé API).

## Pré-requis
```bash
# Dézippe rapports_techniques_2024.zip dans  ../data/rapports_techniques_2024/
# Décommente le bloc bonus de requirements.txt puis :
pip install chromadb sentence-transformers
```

## 1. Repérer les rapports (fourni)

In [ ]:
from pathlib import Path

RAPPORTS = Path("../data/rapports_techniques_2024")
EMBED_MODEL = "all-MiniLM-L6-v2"  # Adapté au corpus ? (regarde la langue des rapports)

fichiers = sorted(RAPPORTS.glob("*.md"))
print(f"{len(fichiers)} rapports trouvés :")
for f in fichiers:
    print(" -", f.name)

## 2. DocumentLoader (fait main) — **TODO**

Écris une fonction qui lit un `.md` et renvoie un dict avec au moins :
`source`, `reference`, `date` (extraits de l'en-tête par regex) et `texte`.
Pas besoin de LangChain.

In [ ]:
import re


def charger_rapport(path: Path) -> dict:
    # TODO : lire le fichier, extraire référence/date, renvoyer le dict
    raise NotImplementedError("Implémente le loader")


# docs = [charger_rapport(f) for f in fichiers]

## 3. Segmentation (chunking) — **TODO**

Implémente **2 stratégies** et **justifie ton choix** :
- **par titre Markdown** (`##`) : chunks cohérents avec la structure ;
- **par taille fixe** (avec recouvrement) : chunks homogènes.

> C'est l'item syllabus « implémentation de techniques de segmentation ».

In [ ]:
def chunk_par_titre(doc: dict) -> list[dict]:
    # TODO : découpe sur les sections `## `, renvoie une liste de chunks
    #        {id, texte, meta:{source, reference, date, section}}
    raise NotImplementedError


def chunk_taille_fixe(doc: dict, taille: int = 500, overlap: int = 80):
    # TODO : fenêtres de `taille` caractères avec recouvrement `overlap`
    raise NotImplementedError


# Compare le nombre de chunks des 2 stratégies et garde-en une pour la suite.

## 4. Embeddings + indexation ChromaDB — **TODO**

Encode chaque chunk avec `SentenceTransformer(EMBED_MODEL)` (local), puis range
les chunks dans une collection ChromaDB **persistante**.

> Pense à **vider** la collection si elle existe déjà (idempotence du notebook).

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# TODO :
# 1) modele = SentenceTransformer(EMBED_MODEL)
# 2) embeddings = modele.encode([... textes des chunks ...]).tolist()
# 3) client = chromadb.PersistentClient(path="./chroma_acerox")
# 4) (re)créer la collection "rapports_acerox" puis col.add(ids, documents,
#    embeddings, metadatas)
raise NotImplementedError("Indexe les chunks dans ChromaDB")

## 5. Interroger par similarité — **TODO**

In [ ]:
def interroger(question: str, k: int = 3) -> None:
    # TODO : encoder la question, col.query(query_embeddings=..., n_results=k),
    #        afficher les passages + métadonnées + distances
    raise NotImplementedError


# Exemples de questions métier à tester :
# interroger("Pourquoi des défauts sur la ligne de laminage ?")
# interroger("Quels capteurs sont peu fiables ?")
# interroger("Risque RGPD avec les données opérateurs de l'ERP ?")

## ✅ Vérification (checklist)

- [ ] J'ai chargé les 5 rapports + leurs métadonnées (loader fait main)
- [ ] J'ai comparé 2 stratégies de **segmentation** et justifié mon choix
- [ ] J'ai indexé les chunks dans ChromaDB avec des embeddings **locaux**
- [ ] Une requête en langage naturel récupère les bons passages
- [ ] J'ai vérifié que le **modèle d'embedding est adapté à la langue du corpus** (les rapports sont en français)
- [ ] J'ai écrit 3 lignes (dans ma note) sur **relationnel vs vectoriel**
- [ ] Je sais où s'arrête la **préparation** vs le **RAG complet** (M7-B2)

## ⭐ Pour aller plus loin
- Filtrer par métadonnée (`where={...}`) pour ne chercher que les rapports récents.
- Brancher un LLM local (Ollama) sur les chunks — mais : est-ce justifié ici ?